# Calculations and demos of the stereographic projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import brentq

In [ ]:
def _invert_one(yi, *, xtol=1e-06):
    if not (0.0 <= yi <= np.pi):
        raise ValueError(f"y must be in [0, Pi], got y={yi}")
    return brentq(lambda x: x - np.sin(x) - yi, a=0.0, b=np.pi, xtol=xtol)
def invert_x_minus_sin_x(y, xtol=1e-06):
    return np.vectorize(_invert_one, otypes=[float])(y, xtol=xtol)

In [ ]:
x_g = np.linspace(0, np.pi, 200)
y = np.linspace(0, np.pi, 100)

In [ ]:
%%timeit -n 500 -r 10
x_p = invert_x_minus_sin_x(y, xtol=1e-06)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

ax.plot(x_g, x_g-np.sin(x_g), label='x - sin(x)',
        color='black', lw=3, zorder=0)
ax.scatter(x_p, y, s=7**2, label='brentq', 
           color='tab:red', marker='x', alpha=0.5, zorder=1)

ax.legend()

plt.show()

## Spherical binner

In [ ]:
from abc import ABC, abstractmethod

In [ ]:
class SphericalBinner(ABC):
    '''TODO
    '''
    @abstractmethod
    def r_limit(self, i):
        '''TODO'''
        raise NotImplementedError
    def r_centroid(self, i):
        '''
        Parameters:
        -----------
        i : int or ndarray
            The index of the bin.
        
        Returns:
        --------
        r_i : float or ndarray
            The centroid of the i-th bin.
        '''
        r_0 = self.r_limit(i)
        r_1 = self.r_limit(i+1)
        return self.centroid(r_0, r_1)
    @staticmethod
    def centroid(r_0, r_1):
        r'''
        Calculates the centroid of a conical frustum along the radius in
        3D space for the i-th bin between r_0 and r_1.
        The formula used is:
        .. math::
            r_i = \frac{1}{4} \frac{r_1^3 - r_0^3}{r_1^2 - r_0^2} + r_0
        where :math:`r_0` and :math:`r_1` are the lower and upper limits
        of the bin.

        Parameters:
        -----------
        r_0 : float or ndarray
            The lower limit of the bin.
        r_1 : float or ndarray
            The upper limit of the bin.

        Returns:
        --------
        r_i : float or ndarray
            The centroid of the i-th bin.

        Notes:
        ------
        The formula is applied in a way to avoid numerical instability
        when :math:`r_0` and :math:`r_1` are very close to each other.
        '''
        nom = (r_1-r_0) * (r_0*r_0 + 2*r_0*r_1 + 3*r_1*r_1)  # r_1^3 - r_0^3
        den = (r_0*r_0 + r_0*r_1 + r_1*r_1)                  # r_1^2 - r_0^2
        return 0.25 * nom / den + r_0

    @staticmethod
    def _invert_one(yi, *, xtol=1e-06):
        '''TODO: Check limits'''
        if not (0.0 <= yi <= np.pi):
            raise ValueError(f"y must be in [0, Pi], got y={yi}")
        return brentq(lambda x: x - np.sin(x) - yi, a=0.0, b=np.pi, xtol=xtol)
    
    def invert_x_minus_sin_x(self, y, xtol=1e-06):
        '''
        Inverts the function :math:`x - \sin(x) - y` using Brent's method.
        The function is defined in the range :math:`[0, \pi]`.

        Parameters:
        -----------
        y : float or ndarray
            The value to be inverted.
        xtol : float
            The tolerance for the root-finding algorithm.
        
        Returns:
        --------
        x : float or ndarray
            The inverted value(s) of :math:`x`.
        '''
        return np.vectorize(self._invert_one, otypes=[float])(y, xtol=xtol)

class SphericalConstantOmega(SphericalBinner):
    r'''
    Radial binner that places equally spaced cuts in the stereographic
    polar half-angle :math:`\omega`.

    The first ``n_bins`` shells have an identical angular width
    :math:`\Delta\omega`. An optional fractional extension
    ``last_cell_size`` lets you append an extra (finite) zone so that
    the final grid point lies comfortably short of the projected
    "horizon" at :math:`\omega=\pi/2`.

    Parameters:
    -----------
    r_4d : float
        The diameter of the 4D sphere.
    n_bins : int
        The number of radial bins to be used.
    last_cell_size : float
        The size of the last non-infinite cell.
    '''
    def __init__(self, r_4d, n_bins, last_cell_size):
        self.r_4d = r_4d
        self.n_bins = n_bins
        self.last_cell_size = last_cell_size
        self.d_omega = np.pi / (2 * (self.n_bins + self.last_cell_size))

    def r_limit(self, i):
        omega = i * self.d_omega
        return self.r_4d * np.tan(omega/2)

class SphericalConstantVolume(SphericalBinner):
    r'''
    Uniformly spaced shells in the Euclidean space with each shell
    enclosing the same Euclidean volume.

    The total volume inside the half-angle :math:`\omega` is given by
    .. math::
        V(\omega)
        =
        \int_0^{\omega} 4 \pi (r_4d \tan(\omega'))^2
        \mathrm{d} (r_4d \tan(\omega'))
        =
        2 \pi r_4d^3 [ x - \sin(x) ]\,, \qquad x \equiv 2 \omega\,.


    Parameters:
    -----------
    r_3d : float
        The radius of the simulation volume in real space.
    r_4d : float
        The diameter of the 4D sphere.
    n_bins : int
        The number of radial bins to be used.
    '''
    def __init__(self, r_3d, r_4d, n_bins):
        self.r_4d = r_4d
        self.n_bins = n_bins
        # Largest possible half-angle, given r = r_4d * tan(omega/2)
        self.omega_max = 2 * np.arctan(r_3d/r_4d)
        # Bin width in the Euclidean space, given x = 2 * omega
        self.bin_width = (2*self.omega_max - np.sin(2*self.omega_max)) / n_bins

    def r_limit(self, i):
        omega = self.invert_x_minus_sin_x(i * self.bin_width) / 2
        return self.r_4d * np.tan(omega/2)

In [ ]:
r_3d = 2000  # [Mph]
r_4d = 105   # [Mph]
n_bins = 224

In [ ]:
last_cell_size = n_bins*np.pi / (2*np.arctan(r_3d/r_4d)) - n_bins
print(f"last_cell_size = {last_cell_size}")

binner_comg = SphericalConstantOmega(r_4d=r_4d, n_bins=n_bins, last_cell_size=last_cell_size)
print(f"binner_comg.d_omega = {binner_comg.d_omega}")
print(f"binner_comg.d_omega * n_bins = {binner_comg.d_omega * n_bins}")
print()
binner_cvol = SphericalConstantVolume(r_4d=r_4d, n_bins=n_bins, r_3d=r_3d)
print(f"binner_cvol.omega_max = {binner_cvol.omega_max}")
print(f"binner_cvol.bin_width = {binner_cvol.bin_width}")
print(f"binner_cvol.bin_width * n_bins = {binner_cvol.bin_width * n_bins}")

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

i = np.arange(0, n_bins)

ax.plot(i, binner_comg.r_limit(i=i), label='SphericalConstantOmega')
ax.plot(i, binner_comg.r_centroid(i=i), label='SphericalConstantOmega centroid')

ax.plot(i, binner_cvol.r_limit(i=i), label='SphericalConstantVolume')
ax.plot(i, binner_cvol.r_centroid(i=i), label='SphericalConstantVolume centroid')

ax.set_xlabel('Bin index')
ax.legend(loc='upper left', fontsize=8)

plt.show()

## Cylindrical binner